# Chapter 16: Gate 3 — Risk Scoring (Reference)

## Learning Objectives

- Compute the Gate 3 risk score from raw PR metadata
- Reproduce the five-row worked example from the chapter
- Show that lowering the threshold can flip a borderline PR's verdict
- Explain why the hard ceiling exists on top of the weighted formula

## Setup

The next cell sets up reproducibility, the `PRA_MODE` fixture/live toggle, and inserts the repo root onto `sys.path` so `pr_automerge` is importable. You should see `PRA_MODE = 'fixture'` printed (unless you've set it to `live`).

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# Reproducibility -- always set before any stochastic operation
RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

# PRA_ environment variables -- fixture mode by default so this notebook runs
# identically for every reader. Set PRA_MODE=live and PRA_REPO=owner/name before
# starting Jupyter to run this against the real sandbox repo instead.
PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")
OUTPUT_DIR = Path(os.environ.get("PRA_OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run in live mode"

print(f"PRA_MODE = {PRA_MODE!r}")
print(f"RANDOM_STATE = {RANDOM_STATE}")

PRA_MODE = 'fixture'
RANDOM_STATE = 42


## 1. Score a Single PR by Hand

The next cell constructs one `PRMetadata` directly and scores it with `evaluate_gate3`. You should see a risk near 66.0 and a PASS verdict, matching the 'small feature' row of Chapter 16's table.

In [2]:
from pr_automerge.models import PRMetadata
from pr_automerge.scoring import evaluate_gate3

pr = PRMetadata(1, "Small feature", "main", "feat/x", 100, 20, 6)
result = evaluate_gate3(pr)
print(result.status.value, result.rationale)

pass risk=66.0 <= threshold=70


## 2. Reproduce the Full Worked Example

The next cell runs every fixture from Chapter 16 Section 8 through `run_worked_example` from `labs/lab_16_risk_scoring.py`. You should see all five rows, ending in two HOLD verdicts.

In [3]:
from labs.lab_16_risk_scoring import run_worked_example
from pr_automerge.scoring import RiskConfig

run_worked_example(RiskConfig())

PR                        Lines  Files  Crit    Risk  Verdict
-------------------------------------------------------------
fix: correct typo in sa       3      1     0     5.9  MERGE
  risk=5.9 <= threshold=70
feat: add greeting help     120      6     0    66.0  MERGE
  risk=66.0 <= threshold=70
ci: tweak gate3 thresho      40      2     1    67.0  MERGE
  risk=67.0 <= threshold=70
refactor: reorganize sa     340     14     0   172.0  HOLD
  risk=172.0 > threshold=70
migrate: restructure sa     900     30     2   510.0  HOLD
  900 lines exceeds the hard ceiling of 500 — never auto-merged regardless of score


## 3. Flip a Borderline Verdict

The next cell re-scores the 'refactor' PR (risk 172.0) at two different thresholds. You should see it HOLD at the default 70, then MERGE once the threshold is loosened to 200 -- the calibration lever Chapter 19 tunes for real.

In [4]:
refactor_pr = PRMetadata(2, "Refactor", "main", "refactor/x", 250, 90, 14)

for threshold in (70.0, 200.0):
    r = evaluate_gate3(refactor_pr, RiskConfig(threshold=threshold))
    print(f"threshold={threshold}: {r.status.value} ({r.rationale})")

threshold=70.0: fail (risk=172.0 > threshold=70)
threshold=200.0: pass (risk=172.0 <= threshold=200)


## Takeaways & Next Steps

This notebook's takeaways are the numbers you just produced above, not abstract claims -- re-read the printed output from each section before moving on.

In [5]:
print(
    "Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo."
)

Re-run this notebook with PRA_MODE=live to see it against the real sandbox repo.


---

📖 **Reading companion:** [Chapter 16: Gate 3 — Risk Scoring](../learning_modules/chapter_16_gate3_risk_scoring.md)
